# 연구 논문용 그래프 생성 노트북

## 목적
이 노트북은 연구 논문에 삽입될 **13개 핵심 그래프**를 생성합니다. 모든 그래프는 인쇄 품질(600 DPI)로 저장되며, 한국어 레이블이 포함됩니다.

## 그래프 구성
### 결과 1: 모델별 분류 성능 비교 (80:20 학습/검증) — RQ1
- **Fig 1-1**: 네 가지 접근법 성능 비교 (ML, BERT, LLM, Hybrid) — 그룹 막대 차트
- **Fig 1-2**: 접근법별 혼동 행렬 (Recall 기반 정규화) — 1x3 히트맵
- **Fig 1-3**: 레이블별 F1 성능 비교 — 그룹 막대 차트
- **Fig 1-4**: KLUE-BERT Cross-Seed 일반화 — 히트맵

### 결과 2: 맥락 변화에 따른 일반화 성능 — RQ2
- **Fig 2-1**: 학습 문항 수 vs 성능 — 라인 차트
- **Fig 2-2**: 유형 전이 히트맵 (ML vs BERT) — 4x4 히트맵
- **Fig 2-3**: 문항별 ML vs BERT vs LLM Kappa — 그룹 막대 차트
- **Fig 2-4**: 학생 기반 분할에 따른 일반화 — 라인 차트
- **Fig 2-5**: 동일 유형 vs 다른 유형 전이 — 덤벨 차트

### 결과 3: 새로운 데이터를 이용한 외부 타당화 — RQ2
- **Fig 3-1**: 학습 데이터 vs 새 데이터 성능 비교 — Before/After 막대 차트
- **Fig 3-2**: 새 데이터 레이블 분포 비교 — 다중 막대 차트
- **Fig 3-3**: 방법 간 일치도 히트맵 — Kappa 히트맵
- **Fig 3-4**: 프롬프트 최적화 효과 — 막대 차트

In [ ]:
# ============================================================
# 공통 설정
# ============================================================
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
from pathlib import Path

# --- 한국어 폰트 설정 (macOS) ---
# Windows: 'Malgun Gothic', Linux: 'NanumGothic' 등으로 변경
mpl.rcParams['font.family'] = 'Apple SD Gothic Neo'
mpl.rcParams['axes.unicode_minus'] = False  # 마이너스 부호 깨짐 방지
mpl.rcParams['font.size'] = 11              # 기본 글꼴 크기

# --- 저장 경로 ---
SAVE_DIR = Path('.')  # 현재 폴더에 저장 (figures_results/)
DPI = 600             # 해상도 (인쇄용: 300, 화면용: 150~200)

# --- 프로젝트 경로 ---
PROJECT = Path('..')

# --- 공통 색상 팔레트 ---
# 순서: ML, BERT, LLM, Hybrid
COLORS = {
    'ML':     '#4C72B0',  # 파란색 계열
    'BERT':   '#DD8452',  # 주황색 계열
    'LLM':    '#55A868',  # 초록색 계열
    'Hybrid': '#C44E52',  # 빨간색 계열
    'Human':  '#2C2C2C',  # 검정색 (사람 라벨링)
}

print('설정 완료')

---
## 결과 1. 모델별 분류 성능 비교 (80:20 학습/검증)

In [ ]:
# ============================================================
# Fig 1-1: 네 가지 접근법 성능 비교 (Grouped Bar Chart)
# ============================================================
# 핵심 그래프: RQ1에 대한 직접적인 답변

fig, ax = plt.subplots(figsize=(10, 6))  # 그래프 크기 조절

# --- 데이터 ---
approaches = ['ML', 'BERT', 'LLM', 'Hybrid']
metrics = ['Accuracy', 'Macro F1', 'Weighted F1', 'Kappa']

# 각 접근법의 [Accuracy, Macro F1, Weighted F1, Kappa]
data = {
    'ML':     [0.768, 0.599, 0.751, 0.692],
    'BERT':   [0.913, 0.839, 0.912, 0.887],
    'LLM':    [0.620, 0.640, 0.622, 0.532],
    'Hybrid': [0.856, 0.848, None,  0.816],  # Weighted F1 없음 → None
}
# 표준편차 (오차 막대)
errors = {
    'ML':     [0.002, 0.011, 0.003, 0.003],
    'BERT':   [0.014, 0.074, 0.017, 0.019],
    'LLM':    [0.004, 0.009, 0.005, 0.005],
    'Hybrid': [0.002, 0.004, None,  0.003],
}

# --- 그리기 ---
x = np.arange(len(metrics))
width = 0.18  # 막대 너비

for i, approach in enumerate(approaches):
    color = COLORS[approach]
    vals = [v if v is not None else 0 for v in data[approach]]
    errs = [e if e is not None else 0 for e in errors[approach]]
    mask = [v is not None for v in data[approach]]
    
    bars = ax.bar(x + (i - 1.5) * width, vals, width,
                  label=approach, color=color, alpha=0.85,
                  yerr=errs, capsize=3, edgecolor='white', linewidth=0.5)
    
    # 값 라벨 표시
    for j, (v, m) in enumerate(zip(vals, mask)):
        if m and v > 0:
            ax.text(x[j] + (i - 1.5) * width, v + errs[j] + 0.012,
                    f'{v:.3f}', ha='center', va='bottom',
                    fontsize=8.5, fontweight='bold')

# --- 꾸미기 ---
ax.set_ylabel('Score')
ax.set_title('네 가지 접근법의 분류 성능 비교 (80:20 학습/검증)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.08)           # y축 범위
ax.legend(loc='upper left', framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.axhline(y=0.8, color='gray', linestyle=':', alpha=0.3)  # 참고선
ax.axhline(y=0.6, color='gray', linestyle=':', alpha=0.3)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig1_1_approach_comparison.png', dpi=DPI, bbox_inches='tight')
plt.show()

**결과 해석 (Fig 1-1):**

그룹 막대 차트(Grouped Bar Chart)는 4개 지표에 대한 네 가지 접근법의 성능을 비교합니다. 오차 막대(error bar)는 3회 반복 실행의 표준편차입니다.

- **BERT가 전 지표에서 최고 성능**: Kappa 0.887로 "almost perfect" 수준 달성 → 인간 평가자 수준의 일치도
- **Hybrid(LLM+BERT)**: Kappa 0.816으로 BERT보다 낮지만 ML과 LLM보다 높음 → 두 접근법의 보완적 결합 효과
- **ML**: Kappa 0.692로 "substantial" 수준 → 전통적 기계학습도 의미 있는 성능 달성
- **LLM**: Kappa 0.532로 "moderate" 수준 → 학습 데이터 없이 프롬프트만으로는 한계
- **시사점**: 라벨링 데이터가 있다면 BERT가 최선, 없다면 LLM+BERT Hybrid 활용 고려

**결과 해석 (Fig 1-2):**

혼동 행렬(Confusion Matrix)은 Recall 기반으로 정규화되어, 각 행(실제 레이블)의 합이 1.0입니다. 대각선 값이 높을수록 해당 레이블의 분류 정확도가 높음을 의미합니다.

- **ML**: Label 0, 1, 6의 대각선 값이 높으나 Label 3(루브릭 평가)에서 분산이 큼
- **BERT**: 모든 대각선 값이 0.7 이상으로 가장 균일한 성능. 특히 Label 3과 5에서 ML/LLM 대비 월등
- **LLM**: Label 0의 Recall=1.00이지만, Label 3은 0.25 미만으로 극히 저조 → LLM이 루브릭 평가를 다른 레이블로 과다 분산 예측

**결과 해석 (Fig 1-3):**

레이블별 F1-Score 비교 막대 차트입니다. 각 레이블에 대해 ML(파랑), BERT(주황), LLM(초록)의 F1을 비교합니다.

- **BERT 우위**: 모든 레이블에서 BERT가 가장 높은 F1을 기록하며, 특히 Label 3, 4, 5(고수준 피드백)에서 격차가 큼
- **LLM의 강점**: Label 0(무관)과 Label 6(대안 제시)에서 상대적으로 높은 F1 → 명확한 언어 패턴이 있는 레이블은 프롬프트 기반 분류가 효과적
- **ML의 약점**: Label 5(구체적 설명)와 Label 6(대안 제시)에서 가장 낮은 F1 → TF-IDF가 고수준 피드백의 의미적 차이를 포착하지 못함

**결과 해석 (Fig 1-4):**

KLUE-BERT Cross-Seed 일반화 히트맵입니다. 3x3 매트릭스의 행은 학습 seed, 열은 테스트 seed이며, 대각선은 같은 seed로 학습/테스트한 결과(검은 테두리)입니다.

- **역전 현상**: 대각선(same-seed) 값이 비대각선(cross-seed)보다 오히려 낮은 경우가 있음 (예: Seed 42 같은 seed=0.904, 다른 seed=0.970/0.976)
- **의미**: BERT 모델이 특정 seed에 과적합되지 않고, 다른 데이터 분할에서도 안정적으로 높은 성능을 유지 → **강건한 일반화** 능력
- **전체 범위**: 0.894~0.986으로 모든 조합이 "almost perfect" 수준

In [ ]:
# ============================================================
# Fig 1-2: 접근법별 혼동 행렬 (ML, BERT, LLM — 1×3 subplot)
# ============================================================
# Recall 기반 정규화: 각 행(실제 레이블)의 합이 1

# --- 예측 데이터 로드 ---
ml_preds   = pd.read_csv(Path('results/ml_predictions.csv'))
bert_preds = pd.read_csv(Path('results/bert_predictions.csv'))
llm_preds  = pd.read_csv(Path('results/llm_predictions.csv'))

labels = list(range(7))  # 0~6
label_names = ['0 무관', '1 인상', '2 정오', '3 평가', '4 오류', '5 설명', '6 대안']
short_names = ['0', '1', '2', '3', '4', '5', '6']  # x축용 간략 이름

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))  # 가로 3개 배치

for ax, (name, df) in zip(axes, [('ML', ml_preds), ('BERT', bert_preds), ('LLM', llm_preds)]):
    # 컬럼명 자동 탐색 (true/label → 실제, run_1/pred → 예측)
    true_col = [c for c in df.columns if 'true' in c.lower() or 'label' in c.lower() or 'actual' in c.lower()]
    pred_col = [c for c in df.columns if 'run_1' in c.lower() or 'pred' in c.lower()]
    
    if not true_col or not pred_col:
        print(f"  {name} 컬럼을 찾을 수 없습니다: {df.columns.tolist()[:10]}")
        continue
    
    y_true = df[true_col[0]]
    y_pred = df[pred_col[0]]
    
    # Recall 기반 정규화 혼동 행렬
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)
    
    # 히트맵 그리기
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1, aspect='auto')
    
    # 셀 값 표시
    for i_r in range(7):
        for j_c in range(7):
            val = cm_norm[i_r, j_c]
            color = 'white' if val > 0.5 else 'black'
            fw = 'bold' if val > 0.3 else 'normal'
            ax.text(j_c, i_r, f'{val:.2f}', ha='center', va='center',
                    fontsize=8, color=color, fontweight=fw)
    
    ax.set_xticks(range(7))
    ax.set_yticks(range(7))
    ax.set_xticklabels(short_names, fontsize=9)
    ax.set_yticklabels(label_names, fontsize=8.5)
    ax.set_xlabel('예측 레이블', fontsize=10)
    if ax == axes[0]:
        ax.set_ylabel('실제 레이블', fontsize=10)
    ax.set_title(f'{name}', fontsize=13, fontweight='bold', pad=8)

plt.suptitle('접근법별 혼동 행렬 (Recall 기반 정규화)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig1_2_confusion_matrices.png', dpi=DPI, bbox_inches='tight')
plt.show()

**결과 해석 (Fig 2-1):**

학습 문항 수(1~7)에 따른 ML 24개 모델의 성능 추이입니다. 주황 실선은 최고 Kappa, 파란 점선은 중앙값, 음영은 25~75 백분위 범위입니다.

- **학습 문항 수 효과**: 1문항(~0.55) -> 7문항(~0.81)으로 문항 증가에 따라 최고 Kappa가 꾸준히 상승
- **수확 체감**: 4문항 이후 향상폭이 둔화 → 약 4개 문항이면 충분한 학습 데이터 다양성 확보
- **빨간 참조선(0.806)**: 전체 8문항 데이터 사용 시의 Kappa → 7문항으로도 전체 데이터에 근접
- **시사점**: 교육 현장에서 4개 문항의 피드백만 라벨링해도 ML 모델의 80% 이상 성능을 달성 가능

**결과 해석 (Fig 2-2):**

유형 전이 히트맵(4x4)은 한 유형으로 학습하여 다른 유형으로 테스트한 Kappa를 보여줍니다. 빨간 점선 테두리가 대각선(동일 유형 전이)입니다.

- **대각선 vs 비대각선**: 대각선이 항상 최고가 아닌 "역전 현상"이 존재 (예: ML에서 실험설계->가설평가(0.730) > 실험설계->실험설계(0.678))
- **ML vs BERT**: BERT가 전반적으로 균일한 색상(안정적 전이)인 반면, ML은 유형 간 격차가 큼
- **가설평가 유형**: 두 접근법 모두에서 대각선 값이 높아(ML 0.752, BERT 0.741), 가장 일반화가 잘 되는 유형
- **시사점**: 문항 유형의 인지적 유사성이 전이 성능에 영향을 미치며, BERT가 유형 간 차이를 더 잘 극복

**결과 해석 (Fig 2-3):**

문항별 ML vs BERT vs LLM Kappa 비교 막대 차트입니다. x축의 괄호 안은 각 문항의 유형을 나타냅니다.

- **BERT 우위**: 모든 8개 문항에서 BERT가 최고 Kappa (0.815~0.927)
- **LLM의 안정성**: 녹색 막대가 비교적 균일한 높이(0.54~0.68) → 문항 유형에 관계없이 일정한 성능을 제공하지만 전체적으로 낮음
- **문항별 차이**: Q6(실험설계, BERT 0.927)가 가장 높고, Q4(예측, BERT 0.815)가 가장 낮음 → 문항 특성이 분류 난이도에 영향
- **LLM 평균선(초록 점선)**: LLM의 문항 간 변동이 작아 수평선에 가까움 → 프롬프트 기반 분류의 안정적이나 제한적인 성능 특성

In [ ]:
# ============================================================
# Fig 1-3: 레이블별 F1 성능 비교 (ML vs BERT vs LLM)
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))

label_display = ['0\n무관', '1\n인상', '2\n정오', '3\n평가', '4\n오류', '5\n설명', '6\n대안']

for idx, (name, df, color) in enumerate([
    ('ML', ml_preds, COLORS['ML']),
    ('BERT', bert_preds, COLORS['BERT']),
    ('LLM', llm_preds, COLORS['LLM']),
]):
    # 컬럼 탐색
    true_col = [c for c in df.columns if 'true' in c.lower() or 'label' in c.lower() or 'actual' in c.lower()]
    pred_col = [c for c in df.columns if 'run_1' in c.lower() or 'pred' in c.lower()]
    if not true_col or not pred_col:
        continue
    
    y_true = df[true_col[0]]
    y_pred = df[pred_col[0]]
    
    # 레이블별 F1 계산
    report = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
    f1s = [report[str(l)]['f1-score'] for l in labels]
    
    x = np.arange(7)
    width = 0.25  # 막대 너비
    bars = ax.bar(x + (idx - 1) * width, f1s, width,
                  label=name, color=color, alpha=0.85, edgecolor='white')
    
    # 값 라벨
    for j, v in enumerate(f1s):
        if v > 0.05:
            ax.text(x[j] + (idx - 1) * width, v + 0.015, f'{v:.2f}',
                    ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.set_ylabel('F1-Score')
ax.set_title('레이블별 F1 성능 비교 (ML vs BERT vs LLM)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xticks(np.arange(7))
ax.set_xticklabels(label_display, fontsize=9)
ax.set_ylim(0, 1.15)
ax.legend(loc='upper right', framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig1_3_label_f1_comparison.png', dpi=DPI, bbox_inches='tight')
plt.show()

**결과 해석 (Fig 2-5):**

덤벨 차트(Dumbbell Chart)는 각 문항에 대해 동일 유형 Kappa(주황 점)와 다른 유형 평균 Kappa(파란 점)를 선으로 연결하여 차이(gap)를 시각화합니다.

- **빨간 연결선**: 동일 유형 > 다른 유형 (유형 일치가 도움이 되는 경우)
- **초록 연결선**: 다른 유형 > 동일 유형 (유형 일치가 도움이 안 되는 경우)
- **ML**: 양방향이 혼재하며, Q7/Q8(가설평가)에서 동일 유형 우위가 가장 뚜렷 → ML은 문항 유형에 따라 전이 패턴이 다양
- **BERT**: 대부분의 문항에서 격차가 작거나 역전 → BERT는 유형에 덜 의존적인 범용 표현 학습
- **시사점**: 학습 데이터 구성 시 유형 다양성이 유형 일치보다 더 중요할 수 있음

In [ ]:
# ============================================================
# Fig 1-4: KLUE-BERT Cross-Seed 일반화 Heatmap
# ============================================================
# 대각선(same-seed)보다 off-diagonal이 높은 "역전 현상"을 시각화

fig, ax = plt.subplots(figsize=(7, 6))

# --- 데이터: KLUE-BERT Macro F1 (Train Seed × Test Seed) ---
cross_seed = np.array([
    [0.9039, 0.9700, 0.9761],  # train seed 42 → test 42, 43, 44
    [0.9734, 0.8936, 0.9846],  # train seed 43
    [0.9803, 0.9859, 0.9409],  # train seed 44
])

seeds = ['Seed 42', 'Seed 43', 'Seed 44']
im = ax.imshow(cross_seed, cmap='YlOrRd', vmin=0.88, vmax=1.0, aspect='auto')

# 셀 값 + 대각선 강조
for i in range(3):
    for j in range(3):
        val = cross_seed[i, j]
        color = 'white' if val > 0.97 else 'black'
        fw = 'bold' if i == j else 'normal'
        ax.text(j, i, f'{val:.4f}', ha='center', va='center',
                fontsize=12, color=color, fontweight=fw)
        # 대각선 셀에 테두리
        if i == j:
            ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                         fill=False, edgecolor='black', linewidth=2.5))

ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(seeds, fontsize=11)
ax.set_yticklabels(seeds, fontsize=11)
ax.set_xlabel('테스트 Seed', fontsize=12)
ax.set_ylabel('학습 Seed', fontsize=12)
ax.set_title('KLUE-BERT Cross-Seed 일반화 (Macro F1)',
             fontsize=13, fontweight='bold', pad=12)
plt.colorbar(im, ax=ax, shrink=0.8, label='Macro F1')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig1_4_bert_cross_seed.png', dpi=DPI, bbox_inches='tight')
plt.show()

**결과 해석 (Fig 3-1):**

Before/After 막대 차트는 학습 데이터(80:20 분할)와 완전히 새로운 데이터에서의 성능 차이를 보여줍니다. 빨간 박스 안의 하락률(%)이 핵심입니다.

- **ML**: F1 0.769 -> 0.149 (81% 하락) — 가장 심각한 성능 저하. TF-IDF 기반 ML이 학습 데이터의 어휘 패턴에 과적합됨
- **BERT**: F1 0.913 -> 0.369 (60% 하락) — 사전학습 지식이 있지만 도메인 이동에 취약
- **LLM**: F1 0.652 -> 0.387 (41% 하락) — 상대적으로 가장 작은 하락률. 프롬프트 기반이라 특정 데이터에 의존하지 않음
- **Hybrid**: F1 0.848 -> 0.430 (49% 하락) — LLM과 BERT의 중간 수준
- **핵심 발견**: 새로운 맥락에서 LLM이 ML/BERT를 역전하는 현상 발생 → 도메인 이동이 심한 상황에서는 학습 기반 모델의 강점이 약점으로 전환

**결과 해석 (Fig 3-2):**

새 데이터에 대한 사람 라벨링과 4가지 모델 예측의 레이블 분포(%)를 비교합니다.

- **ML의 Label 0 편향**: ML이 53.4%를 Label 0(무관)으로 예측하여 실제 분포(9.8%)와 극도로 불일치 → ML이 새로운 어휘 패턴을 "무관"으로 잘못 분류
- **LLM의 균형**: LLM 분포가 사람 라벨링과 가장 유사한 형태 → 프롬프트 기반 분류가 도메인 이동에 더 강건
- **Hybrid의 보완 효과**: LLM 1차 분류 후 BERT 재분류로 Label 1(막연한 인상)이 사람 분포와 유사하게 조정됨
- **시사점**: 도메인 이동 상황에서 ML은 레이블 분포를 심각하게 왜곡하므로 주의 필요

**결과 해석 (Fig 3-3):**

방법 간 일치도 히트맵(Cohen's Kappa)은 새 데이터에 대한 4가지 방법의 예측이 서로 얼마나 일치하는지 보여줍니다.

- **ML의 고립**: ML과 다른 방법 간 Kappa가 0.07~0.14로 극히 낮음 → ML이 새 데이터에서 완전히 다른 판단을 내림 (Label 0 편향 때문)
- **LLM-Hybrid 높은 일치**: 0.831로 가장 높은 일치도 → Hybrid의 기반이 LLM이므로 자연스러운 결과
- **BERT-Hybrid 중간 일치**: 0.525 → BERT 재분류가 Hybrid의 일부 예측을 수정
- **시사점**: 도메인 이동 상황에서 ML의 판단은 다른 방법들과 크게 괴리되므로, 새로운 데이터에 ML을 직접 적용하는 것은 부적절

**결과 해석 (Fig 3-4):**

프롬프트 최적화에 따른 새 데이터 성능 향상을 보여줍니다. 빗금(hatch) 막대는 원래 프롬프트, 단색은 최적화 프롬프트입니다.

- **LLM**: 원래 Kappa 0.318 -> 최적화 후 0.541 (+0.223 향상) — 프롬프트 수정만으로 70% 성능 향상
- **Hybrid**: 원래 Kappa 0.359 -> 최적화 후 0.565 (+0.206 향상)
- **핵심 발견**: 새로운 데이터에 맞는 프롬프트 최적화가 LLM/Hybrid 성능을 크게 향상시킴
- **실용적 시사점**: 완전히 새로운 맥락에서는 학습 데이터 없이도 프롬프트 엔지니어링만으로 "moderate~substantial" 수준의 분류가 가능 → 학습 데이터 구축이 어려운 상황에서의 대안

---
## 결과 2. 맥락 변화에 따른 일반화 성능

In [ ]:
# ============================================================
# Fig 2-1: 학습 문항 수 vs 성능 (Line Chart)
# ============================================================
# 254개 문항 조합 × 24개 ML 모델 실험 결과 요약

fig, ax = plt.subplots(figsize=(9, 6))

# --- 데이터 로드 ---
combo_df = pd.read_csv(
    PROJECT / '2_ml/results/generalization/question_combo/qc_all_results_question_combo_parallel24.csv'
)
combo_df['N_Train_Qs'] = combo_df['N_Train_Qs'].astype(int)

# 학습 문항 수별 집계
best_by_n = combo_df.groupby('N_Train_Qs').agg(
    Best_Kappa=('Test_Kappa', 'max'),
    Mean_Kappa=('Test_Kappa', 'mean'),
    Median_Kappa=('Test_Kappa', 'median'),
).reset_index()

# 25, 75 백분위
pct = combo_df.groupby('N_Train_Qs')['Test_Kappa'].quantile([0.25, 0.75]).unstack()
pct.columns = ['Q25', 'Q75']
best_by_n = best_by_n.merge(pct, left_on='N_Train_Qs', right_index=True)

# --- 그리기 ---
x = best_by_n['N_Train_Qs']
ax.plot(x, best_by_n['Best_Kappa'], 'o-', color='#DD8452',
        linewidth=2.5, markersize=8, label='최고 Kappa', zorder=5)
ax.plot(x, best_by_n['Median_Kappa'], 's--', color='#4C72B0',
        linewidth=2, markersize=7, label='중앙값 Kappa', zorder=4)
ax.fill_between(x, best_by_n['Q25'], best_by_n['Q75'],
                alpha=0.15, color='#4C72B0', label='25~75 백분위')

# 전체 데이터 성능 참고선
ax.axhline(y=0.806, color='red', linestyle=':', alpha=0.6,
           linewidth=1.5, label='전체 데이터 Kappa (0.806)')

# 값 라벨
for _, row in best_by_n.iterrows():
    ax.text(row['N_Train_Qs'], row['Best_Kappa'] + 0.012,
            f'{row["Best_Kappa"]:.3f}', ha='center', fontsize=8.5,
            fontweight='bold', color='#DD8452')

# --- 꾸미기 ---
ax.set_xlabel('학습 문항 수', fontsize=12)
ax.set_ylabel("Cohen's Kappa", fontsize=12)
ax.set_title('학습 문항 수에 따른 분류 성능 변화 (ML 24개 모델)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(range(1, 8))
ax.set_xlim(0.5, 7.5)
ax.set_ylim(0.2, 0.9)  # y축 범위 조절
ax.legend(loc='lower right', framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig2_1_nq_vs_performance.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Fig 2-2: 유형 전이 Heatmap (ML vs BERT, 4×4)
# ============================================================
# 행: 학습 유형, 열: 테스트 유형 → 대각선이 동일 유형

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

short_types = ['자료해석\n·결론', '실험설계', '예측', '가설평가']

# --- 데이터: 유형별 평균 Kappa ---
# ML (C3_Optimal3_SoftVoting)
ml_matrix = np.array([
    [0.609, 0.565, 0.569, 0.444],
    [0.582, 0.678, 0.626, 0.730],
    [0.524, 0.507, 0.541, 0.548],
    [0.442, 0.617, 0.618, 0.752],
])
# BERT (KLUE-BERT)
bert_matrix = np.array([
    [0.631, 0.698, 0.627, 0.666],
    [0.643, 0.694, 0.648, 0.717],
    [0.536, 0.637, 0.584, 0.649],
    [0.508, 0.633, 0.585, 0.741],
])

vmin, vmax = 0.4, 0.8  # 색상 범위 (조절 가능)

for ax, mat, title in zip(axes, [ml_matrix, bert_matrix],
                           ['ML (C3_Optimal3_SoftVoting)', 'BERT (KLUE-BERT)']):
    im = ax.imshow(mat, cmap='YlGnBu', vmin=vmin, vmax=vmax, aspect='auto')
    
    for i in range(4):
        for j in range(4):
            val = mat[i, j]
            color = 'white' if val > 0.65 else 'black'
            fw = 'bold' if i == j else 'normal'
            ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                    fontsize=11, color=color, fontweight=fw)
            # 대각선(동일 유형) 강조
            if i == j:
                ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                             fill=False, edgecolor='red', linewidth=2.5, linestyle='--'))
    
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(short_types, fontsize=9)
    ax.set_yticklabels(short_types, fontsize=9)
    ax.set_xlabel('테스트 유형', fontsize=11)
    if ax == axes[0]:
        ax.set_ylabel('학습 유형', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=8)

plt.suptitle('문항 유형 간 전이 성능 (Kappa)', fontsize=14, fontweight='bold', y=1.02)
fig.colorbar(im, ax=axes, shrink=0.8, label="Cohen's Kappa", pad=0.02)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig2_2_type_transfer_heatmap.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Fig 2-3: 문항별 ML vs BERT vs LLM Kappa 비교
# ============================================================
# LLM은 학습 없이 프롬프트 기반 → 문항 간 변동 작음을 보여줌

fig, ax = plt.subplots(figsize=(13, 6.5))

questions = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8']
q_types = ['자료해석·결론', '실험설계', '자료해석·결론', '예측',
           '예측', '실험설계', '가설평가', '가설평가']

# --- 데이터 ---
# 각 문항에서 전체 데이터로 학습했을 때의 해당 문항 Kappa
llm_kappa      = [0.677, 0.678, 0.620, 0.537, 0.579, 0.619, 0.650, 0.569]
ml_best_kappa  = [0.765, 0.787, 0.811, 0.713, 0.850, 0.814, 0.819, 0.818]
bert_kappa     = [0.865, 0.894, 0.835, 0.815, 0.864, 0.927, 0.909, 0.875]

x = np.arange(len(questions))
width = 0.25

ax.bar(x - width, ml_best_kappa, width, label='ML (최적)', color=COLORS['ML'], alpha=0.85, edgecolor='white')
ax.bar(x,         bert_kappa,    width, label='BERT',      color=COLORS['BERT'], alpha=0.85, edgecolor='white')
ax.bar(x + width, llm_kappa,    width, label='LLM',       color=COLORS['LLM'], alpha=0.85, edgecolor='white')

# 값 라벨
for bars_data, offset in [(ml_best_kappa, -width), (bert_kappa, 0), (llm_kappa, width)]:
    for j, v in enumerate(bars_data):
        ax.text(x[j] + offset, v + 0.008, f'{v:.2f}',
                ha='center', va='bottom', fontsize=7.5, fontweight='bold')

# x축: Q번호 + 유형을 2줄로
xlabels = [f'{q}\n({t})' for q, t in zip(questions, q_types)]
ax.set_xticks(x)
ax.set_xticklabels(xlabels, fontsize=9)

ax.set_ylabel("Cohen's Kappa", fontsize=12)
ax.set_title('문항별 분류 성능 비교 (ML vs BERT vs LLM)',
             fontsize=14, fontweight='bold', pad=12)
ax.set_ylim(0, 1.08)
ax.legend(loc='upper left', framealpha=0.9, fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# LLM 평균선 (일관성 강조)
ax.axhline(y=np.mean(llm_kappa), color=COLORS['LLM'], linestyle=':', alpha=0.4)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig2_3_question_kappa_comparison.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Fig 2-4: 학생 기반 분할에 따른 일반화 성능
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5.5))

# --- 데이터 ---
ratios     = [40, 50, 60, 70, 80]      # 학습 학생 비율 (%)
mean_kappa = [0.614, 0.615, 0.618, 0.627, 0.650]
best_kappa = [0.737, 0.748, 0.739, 0.793, 0.776]

# --- 그리기 ---
ax.plot(ratios, best_kappa, 'o-', color='#DD8452',
        linewidth=2.5, markersize=8, label='최고 Kappa', zorder=5)
ax.plot(ratios, mean_kappa, 's--', color='#4C72B0',
        linewidth=2, markersize=7, label='평균 Kappa', zorder=4)

# 값 라벨
for r, bk, mk in zip(ratios, best_kappa, mean_kappa):
    ax.text(r, bk + 0.012, f'{bk:.3f}', ha='center', fontsize=8.5,
            fontweight='bold', color='#DD8452')
    ax.text(r, mk - 0.020, f'{mk:.3f}', ha='center', fontsize=8.5, color='#4C72B0')

# 전체 데이터 참고선
ax.axhline(y=0.806, color='red', linestyle=':', alpha=0.5,
           linewidth=1.5, label='전체 데이터 Kappa (0.806)')

ax.set_xlabel('학습 학생 비율 (%)', fontsize=12)
ax.set_ylabel("Cohen's Kappa", fontsize=12)
ax.set_title('학생 기반 분할에 따른 일반화 성능',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(ratios)
ax.set_xlim(35, 85)
ax.set_ylim(0.5, 0.85)
ax.legend(loc='lower right', framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig2_4_student_ratio_performance.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Fig 2-5: 동일 유형 vs 다른 유형 전이 성능 (Dumbbell Chart)
# ============================================================
# 점 2개를 선으로 연결 → 갭이 큰 문항일수록 유형 특수적

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

questions_db = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8']
q_types_db = ['자료해석', '실험설계', '자료해석', '예측',
              '예측', '실험설계', '가설평가', '가설평가']

# --- 데이터: [동일유형 Kappa, 다른유형 평균 Kappa] ---
# ML (C3_Optimal3_SoftVoting)
ml_same  = [0.603, 0.754, 0.615, 0.447, 0.635, 0.602, 0.761, 0.744]
ml_cross = [0.553, 0.652, 0.499, 0.452, 0.601, 0.640, 0.543, 0.575]
# BERT (KLUE-BERT)
bert_same  = [0.632, 0.676, 0.630, 0.619, 0.549, 0.713, 0.738, 0.745]
bert_cross = [0.680, 0.665, 0.648, 0.625, 0.590, 0.674, 0.590, 0.561]

for ax, same, cross, title in zip(
    axes,
    [ml_same, bert_same],
    [ml_cross, bert_cross],
    ['ML (C3_Optimal3_SoftVoting)', 'BERT (KLUE-BERT)'],
):
    y = np.arange(len(questions_db))
    
    # 연결 선 (갭 방향에 따라 색상 변경)
    for i in range(len(questions_db)):
        color = '#C44E52' if same[i] > cross[i] else '#55A868'  # 빨강: 하락, 초록: 상승
        ax.plot([cross[i], same[i]], [i, i], '-', color=color, linewidth=2, alpha=0.6)
    
    # 점
    ax.scatter(same, y, s=100, color='#DD8452', zorder=5,
              label='동일유형 Kappa', edgecolors='white', linewidth=1)
    ax.scatter(cross, y, s=100, color='#4C72B0', zorder=5,
              label='다른유형 평균 Kappa', edgecolors='white', linewidth=1)
    
    # 차이 라벨
    for i in range(len(questions_db)):
        delta = same[i] - cross[i]
        sign = '+' if delta >= 0 else ''
        ax.text(max(same[i], cross[i]) + 0.015, i,
                f'Δ={sign}{delta:.3f}', va='center', fontsize=8,
                color='#C44E52' if delta > 0.05 else 'gray')
    
    ax.set_yticks(y)
    ax.set_yticklabels([f'{q} ({t})' for q, t in zip(questions_db, q_types_db)], fontsize=9)
    ax.set_xlabel("Cohen's Kappa", fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=8)
    ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(0.35, 0.85)
    ax.invert_yaxis()  # Q1이 맨 위
    ax.axvline(x=0.6, color='gray', linestyle=':', alpha=0.3)

plt.suptitle('동일 유형 vs 다른 유형 전이 성능 비교',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig2_5_same_vs_cross_type.png', dpi=DPI, bbox_inches='tight')
plt.show()

---
## 결과 3. 새로운 데이터를 이용한 외부 타당화

In [ ]:
# ============================================================
# Fig 3-1: 학습 데이터 vs 새로운 데이터 성능 비교 (Before/After)
# ============================================================
# 도메인 이동(domain shift)의 심각성을 직관적으로 보여줌

fig, ax = plt.subplots(figsize=(10, 6))

approaches = ['ML', 'BERT', 'LLM', 'Hybrid']
train_f1 = [0.769, 0.913, 0.652, 0.848]  # 학습 데이터 최적 F1 Macro
new_f1   = [0.149, 0.369, 0.387, 0.430]  # 새 데이터 F1 Macro

x = np.arange(len(approaches))
width = 0.35

ax.bar(x - width/2, train_f1, width, label='학습 데이터 (80:20)',
       color=COLORS['ML'], alpha=0.85, edgecolor='white')
ax.bar(x + width/2, new_f1, width, label='새로운 데이터',
       color=COLORS['Hybrid'], alpha=0.85, edgecolor='white')

# 값 라벨 + 하락률 표시
for i in range(len(approaches)):
    ax.text(x[i] - width/2, train_f1[i] + 0.015, f'{train_f1[i]:.3f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold', color=COLORS['ML'])
    ax.text(x[i] + width/2, new_f1[i] + 0.015, f'{new_f1[i]:.3f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold', color=COLORS['Hybrid'])
    
    # 하락률 박스
    drop = (train_f1[i] - new_f1[i]) / train_f1[i] * 100
    mid_y = (train_f1[i] + new_f1[i]) / 2
    ax.annotate(f'↓{drop:.0f}%', xy=(x[i], mid_y), fontsize=10, fontweight='bold',
                ha='center', va='center', color='#8B0000',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFE0E0',
                          edgecolor=COLORS['Hybrid'], alpha=0.8))

ax.set_ylabel('F1 Macro', fontsize=12)
ax.set_title('학습 데이터 vs 새로운 데이터 성능 비교',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(approaches, fontsize=11)
ax.set_ylim(0, 1.08)
ax.legend(loc='upper right', framealpha=0.9, fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig3_1_before_after_performance.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Fig 3-2: 새 데이터 레이블 분포 비교 (사람 vs 4가지 모델)
# ============================================================
# ML의 Label 0 편향, Hybrid의 균형 잡힌 분포를 시각화

fig, ax = plt.subplots(figsize=(14, 6))

label_names_dist = ['0 무관', '1 인상', '2 정오', '3 평가', '4 오류', '5 설명', '6 대안']

# --- 비율 데이터 (%) ---
human  = [9.8,  37.3, 3.1,  13.5, 11.9, 12.4, 11.9]
ml     = [53.4, 22.3, 0.5,  2.1,  9.3,  4.1,  8.3]
bert   = [2.6,  31.1, 4.1,  5.2,  18.1, 29.0, 9.8]
llm    = [13.5, 38.3, 2.6,  1.0,  18.1, 8.3,  18.1]
hybrid = [13.5, 40.9, 2.6,  3.6,  8.3,  12.4, 18.7]

x = np.arange(len(label_names_dist))
width = 0.15  # 5개 바 → 좁은 너비

names  = ['사람',   'ML',   'BERT',   'LLM',   'Hybrid']
datas  = [human,    ml,     bert,     llm,     hybrid]
colors = [COLORS['Human'], COLORS['ML'], COLORS['BERT'], COLORS['LLM'], COLORS['Hybrid']]

for i, (name, data, color) in enumerate(zip(names, datas, colors)):
    offset = (i - 2) * width
    ax.bar(x + offset, data, width, label=name, color=color, alpha=0.85, edgecolor='white')

ax.set_ylabel('비율 (%)', fontsize=12)
ax.set_title('새로운 데이터: 사람 라벨링 vs 모델 예측 레이블 분포',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(x)
ax.set_xticklabels(label_names_dist, fontsize=10)
ax.set_ylim(0, 60)
ax.legend(loc='upper right', framealpha=0.9, ncol=5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig3_2_label_distribution.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Fig 3-3: 방법 간 일치도 Heatmap (Cohen's Kappa)
# ============================================================
# ML이 다른 방법과 극히 낮은 일치도를 보이는 패턴

fig, ax = plt.subplots(figsize=(7, 6))

methods = ['ML', 'BERT', 'LLM', 'Hybrid']
agreement = np.array([
    [1.000, 0.068, 0.140, 0.140],
    [0.068, 1.000, 0.366, 0.525],
    [0.140, 0.366, 1.000, 0.831],
    [0.140, 0.525, 0.831, 1.000],
])

im = ax.imshow(agreement, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')

for i in range(4):
    for j in range(4):
        val = agreement[i, j]
        color = 'white' if (val > 0.7 or val < 0.15) else 'black'
        if i == j:
            ax.text(j, i, '—', ha='center', va='center', fontsize=14, color='gray')
        else:
            fw = 'bold' if val > 0.5 else 'normal'
            ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                    fontsize=12, color=color, fontweight=fw)

ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(methods, fontsize=11)
ax.set_yticklabels(methods, fontsize=11)
ax.set_title("새로운 데이터: 방법 간 일치도 (Cohen's Kappa)",
             fontsize=13, fontweight='bold', pad=12)
plt.colorbar(im, ax=ax, shrink=0.8, label="Cohen's Kappa")

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig3_3_inter_method_agreement.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Fig 3-4: 프롬프트 최적화에 따른 새 데이터 성능 향상
# ============================================================
# 빗금(hatch) = 원래 프롬프트, 단색 = 최적화 프롬프트

fig, ax = plt.subplots(figsize=(9, 5.5))

methods_p = ['LLM', 'Hybrid']

# --- 데이터 ---
original_kappa = [0.318, 0.359]   # 원래 프롬프트
new_kappa      = [0.541, 0.565]   # 최적화 프롬프트
original_f1    = [0.387, 0.430]
new_f1         = [0.585, 0.590]

x = np.arange(len(methods_p))
width = 0.18

# Kappa 막대 (파랑 계열)
ax.bar(x - width*1.5, original_kappa, width, label='원래 프롬프트 (Kappa)',
       color=COLORS['ML'], alpha=0.6, edgecolor='white', hatch='//')
ax.bar(x - width*0.5, new_kappa,      width, label='최적화 프롬프트 (Kappa)',
       color=COLORS['ML'], alpha=0.9, edgecolor='white')

# F1 막대 (주황 계열)
ax.bar(x + width*0.5, original_f1, width, label='원래 프롬프트 (F1)',
       color=COLORS['BERT'], alpha=0.6, edgecolor='white', hatch='//')
ax.bar(x + width*1.5, new_f1,      width, label='최적화 프롬프트 (F1)',
       color=COLORS['BERT'], alpha=0.9, edgecolor='white')

# 값 라벨
all_bars = [
    (original_kappa, x - width*1.5),
    (new_kappa,      x - width*0.5),
    (original_f1,    x + width*0.5),
    (new_f1,         x + width*1.5),
]
for vals, positions in all_bars:
    for pos, v in zip(positions, vals):
        ax.text(pos, v + 0.01, f'{v:.3f}', ha='center', va='bottom',
                fontsize=8.5, fontweight='bold')

# 향상 폭 표시
for i in range(len(methods_p)):
    delta_k = new_kappa[i] - original_kappa[i]
    delta_f = new_f1[i] - original_f1[i]
    ax.annotate(f'+{delta_k:.3f}', xy=(x[i] - width, max(new_kappa[i], original_kappa[i]) + 0.06),
                fontsize=9, ha='center', color='#003380', fontweight='bold')
    ax.annotate(f'+{delta_f:.3f}', xy=(x[i] + width, max(new_f1[i], original_f1[i]) + 0.06),
                fontsize=9, ha='center', color='#8B4500', fontweight='bold')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('프롬프트 최적화에 따른 새 데이터 성능 향상',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(x)
ax.set_xticklabels(methods_p, fontsize=12)
ax.set_ylim(0, 0.75)
ax.legend(loc='upper left', framealpha=0.9, fontsize=9, ncol=2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig3_4_prompt_optimization.png', dpi=DPI, bbox_inches='tight')
plt.show()

---
## 수정 가이드

| 항목 | 위치 | 예시 |
|------|------|------|
| **그래프 크기** | `figsize=(가로, 세로)` | `figsize=(12, 8)` |
| **해상도** | 맨 위 `DPI` 변수 | `DPI = 300` (인쇄용) |
| **색상** | 맨 위 `COLORS` 딕셔너리 | `'ML': '#1f77b4'` |
| **폰트 크기** | `fontsize=` 파라미터 | `fontsize=14` |
| **y축 범위** | `ax.set_ylim(min, max)` | `ax.set_ylim(0.3, 1.0)` |
| **제목** | `ax.set_title(...)` | 한국어/영어 전환 |
| **범례 위치** | `ax.legend(loc=...)` | `'upper right'`, `'lower left'` |
| **저장 형식** | 파일 확장자 변경 | `.pdf`, `.svg` (벡터) |
| **한국어 폰트** | 맨 위 `font.family` | Windows: `'Malgun Gothic'` |